# Camada Gold — Azure Databricks

Lê as tabelas **Silver** (`tech_challenge_fase2.silver`) e publica as tabelas analíticas em `tech_challenge_fase2.gold` (Tarefa 8 do desafio).

| Tabela Gold | Grão | Para que serve |
|---|---|---|
| `indicador_municipio` | município × ano × série × rede | Taxa de alfabetização oficial e **recalculada dos microdados** (ponderada pelo peso, só alunos avaliados) |
| `resultado_vs_meta` | nível (município, UF, Brasil) × ano × recorte | Resultado × meta, com **três estados**: `atingiu`, `nao_atingiu`, `sem_meta` |
| `evolucao_temporal` | UF e Brasil × ano | Série histórica e variação em pontos percentuais |
| `validacao_cruzada_brasil` | ano | Taxa recalculada dos microdados × taxa publicada |
| `features_alfabetizacao` | município × série × rede (features em *t*, alvo em *t+1*) | Base para o modelo da Fase 3, sem vazamento de alvo |

**Cuidados com as armadilhas do enunciado**
1. **Denominador:** a taxa recalculada usa só `avaliado = true` e o `peso_aluno`, nunca o total de registros.
2. **Ausência de meta ≠ não atingiu:** `sem_meta` é um terceiro estado explícito.
3. **Mesmo recorte:** resultado só é comparado com a meta da mesma rede (mapeamento fixo abaixo, com checagem de que nenhuma rede de meta ficou sem mapeamento).
4. **Join que não casa vira nulo:** a taxa de correspondência é medida e **bloqueia** a publicação abaixo do limiar.
5. **Nada de nulo → zero.** Nulo continua nulo.
6. **Sem projeção até 2030:** a série tem poucos pontos, então nenhuma extrapolação é publicada.

**Qualidade:** cada tabela é gravada em `_staging_<tabela>`, passa pelas regras e só então é promovida. As regras **bloqueantes** interrompem a execução; o resultado de todas fica em `gold._quality_results`.

## Configuração do ambiente

In [ ]:
import uuid
from datetime import datetime, timezone
from functools import reduce

from pyspark.sql import functions as F
from pyspark.sql.types import (BooleanType, DoubleType, StringType,
                               StructField, StructType, TimestampType)

dbutils.widgets.text("CATALOGO", "tech_challenge_fase2", "Catálogo")
dbutils.widgets.text("SILVER_SCHEMA", "silver", "Schema da Silver")
dbutils.widgets.text("GOLD_SCHEMA", "gold", "Schema da Gold")
dbutils.widgets.text("LIMIAR_JOIN", "0.95", "Limiar mínimo da taxa de correspondência dos joins (0 a 1)")
dbutils.widgets.text("TOLERANCIA_PP", "0.1", "Tolerância (pontos percentuais) entre taxa recalculada e oficial")


def variavel(nome, padrao=None):
    """Lê um parâmetro do notebook. Valor vazio conta como 'não definido'."""
    valor = dbutils.widgets.get(nome) or padrao
    if valor is None:
        raise RuntimeError(f"Parâmetro {nome} não definido. Preencha o widget no topo do notebook.")
    return valor


CATALOGO = variavel("CATALOGO")
SILVER_SCHEMA = variavel("SILVER_SCHEMA")
GOLD_SCHEMA = variavel("GOLD_SCHEMA")
LIMIAR_JOIN = float(variavel("LIMIAR_JOIN"))
TOLERANCIA_PP = float(variavel("TOLERANCIA_PP"))
FORMATO_TABELA = "delta"

print(f"Origem:  {CATALOGO}.{SILVER_SCHEMA}")
print(f"Destino: {CATALOGO}.{GOLD_SCHEMA}")
print(f"Limiar de correspondência dos joins: {LIMIAR_JOIN:.0%} | tolerância recalculada x oficial: {TOLERANCIA_PP} p.p.")

## Regras de negócio fixas

**Mapeamento rede da meta → código de rede.** As tabelas de meta trazem `rede` como texto (`Municipal`, `Pública`) e o `dicionario` não cobre essas tabelas. O mapeamento abaixo foi **conferido nos dados**: a taxa de `meta_alfabetizacao_municipio` (`Municipal`) é igual à de `municipio` com `rede = 3`, e a de `meta_alfabetizacao_uf` (`Pública`) é igual à de `uf` com `rede = 5` (diferenças só de arredondamento). Na execução, a célula de regras mede essa consistência de novo.

**Composição das redes agregadas.** O `dicionario` descreve `5` como *Pública (Estadual e Municipal)* e `0` como *Total (Federal, Estadual, Municipal e Privada)*. Para recalcular a taxa desses recortes a partir dos microdados, `5 = 2 + 3` e `0 = 1 + 2 + 3 + 4`.

In [ ]:
REDE_META_PARA_CODIGO = {"Municipal": "3", "Pública": "5"}
COMPOSICAO_REDE = {"5": ["2", "3"], "0": ["1", "2", "3", "4"]}

ANOS_META = list(range(2024, 2031))
COLUNAS_META = [f"meta_alfabetizacao_{ano}" for ano in ANOS_META]
NIVEIS = [f"proporcao_aluno_nivel_{n}" for n in range(0, 9)]

## Funções auxiliares

In [ ]:
def tabela_silver(nome):
    return f"`{CATALOGO}`.`{SILVER_SCHEMA}`.`{nome}`"


def tabela_gold(nome):
    return f"`{CATALOGO}`.`{GOLD_SCHEMA}`.`{nome}`"


def codigo_rede_meta(coluna="rede_descricao"):
    """Traduz o texto de rede das tabelas de meta para o código de rede. Texto fora do mapa fica nulo (e a regra bloqueante acusa)."""
    expr = F.lit(None).cast("string")
    for texto, codigo in REDE_META_PARA_CODIGO.items():
        expr = F.when(F.col(coluna) == texto, F.lit(codigo)).otherwise(expr)
    return expr


def meta_do_ano(coluna_ano="ano", deslocamento=0):
    """Escolhe a coluna meta_alfabetizacao_<ano + deslocamento>. Ano sem coluna de meta (ex.: 2023) fica nulo."""
    return F.coalesce(*[
        F.when(F.col(coluna_ano) + deslocamento == a, F.col(f"meta_alfabetizacao_{a}"))
        for a in ANOS_META
    ])


def aplicar_meta(df, coluna_taxa="taxa_alfabetizacao"):
    """Compara a taxa com a meta do mesmo ano, com TRÊS estados (ausência de meta não é fracasso)."""
    return (
        df.withColumn("meta_alfabetizacao_ano", meta_do_ano())
        .withColumn(
            "status_meta",
            F.when(F.col("meta_alfabetizacao_ano").isNull() | F.col(coluna_taxa).isNull(), "sem_meta")
            .when(F.col(coluna_taxa) >= F.col("meta_alfabetizacao_ano"), "atingiu")
            .otherwise("nao_atingiu"),
        )
        .withColumn("distancia_ate_meta_pp", F.round(F.col(coluna_taxa) - F.col("meta_alfabetizacao_ano"), 4))
        .withColumn("distancia_ate_meta_2030_pp", F.round(F.col(coluna_taxa) - F.col("meta_alfabetizacao_2030"), 4))
    )


def taxa_calculada_microdados(alunos, por=("id_municipio",)):
    """Taxa de alfabetização (%) recalculada dos microdados: ponderada pelo peso amostral, só alunos avaliados."""
    base = (
        alunos.where(F.col("avaliado") & F.col("peso_aluno").isNotNull())
        .select("ano", *por, "serie", "rede", "peso_aluno",
                F.col("alfabetizado_avaliado").cast("int").alias("alf"))
    )
    recortes = [base]
    for codigo, composicao in COMPOSICAO_REDE.items():
        recortes.append(base.where(F.col("rede").isin(composicao)).withColumn("rede", F.lit(codigo)))
    todos = reduce(lambda a, b: a.unionByName(b), recortes)
    return todos.groupBy("ano", *por, "serie", "rede").agg(
        F.count(F.lit(1)).alias("alunos_avaliados"),
        F.sum("peso_aluno").alias("soma_peso_avaliados"),
        F.round(100 * F.sum(F.col("peso_aluno") * F.col("alf")) / F.sum("peso_aluno"), 4)
        .alias("taxa_alfabetizacao_calculada"),
    )

## Construção das tabelas Gold (uma função por tabela)

Cada função recebe DataFrames da Silver e devolve o DataFrame da Gold. Nenhuma grava nada.

In [ ]:
def construir_indicador_municipio(municipio, alunos):
    """Município × ano × série × rede: taxa oficial (Silver `municipio`) + taxa recalculada dos microdados (`alunos`)."""
    calculada = taxa_calculada_microdados(alunos)
    oficial = municipio.select(
        "ano", "id_municipio", "serie", "serie_descricao", "rede", "rede_descricao",
        F.col("taxa_alfabetizacao").alias("taxa_alfabetizacao_oficial"),
        F.col("media_portugues").alias("media_portugues_oficial"),
        *NIVEIS,
    )
    df = oficial.join(calculada, ["ano", "id_municipio", "serie", "rede"], "left")
    df = (
        df.withColumn("diferenca_pp", F.round(F.col("taxa_alfabetizacao_calculada") - F.col("taxa_alfabetizacao_oficial"), 4))
        .withColumn("dentro_tolerancia",
                    F.when(F.col("diferenca_pp").isNull(), F.lit(None).cast("boolean"))
                    .otherwise(F.abs(F.col("diferenca_pp")) <= TOLERANCIA_PP))
    )
    return df.select(
        "ano", "id_municipio", "serie", "serie_descricao", "rede", "rede_descricao",
        "taxa_alfabetizacao_oficial", "taxa_alfabetizacao_calculada", "diferenca_pp", "dentro_tolerancia",
        "alunos_avaliados", "soma_peso_avaliados", "media_portugues_oficial", *NIVEIS,
    )


def construir_resultado_vs_meta(municipio, uf, meta_brasil, meta_uf, meta_municipio):
    """Resultado × meta nos três níveis, sempre no mesmo recorte de rede (municipal = rede 3; UF e Brasil = rede 5)."""
    colunas = ["nivel", "ano", "id_municipio", "sigla_uf", "serie", "rede", "rede_descricao",
               "taxa_alfabetizacao", *COLUNAS_META]
    nulo = F.lit(None).cast("string")

    mm = meta_municipio.withColumn("rede", codigo_rede_meta()).select("ano", "id_municipio", "rede", *COLUNAS_META)
    nivel_municipio = (
        municipio.where(F.col("rede") == "3")
        .join(mm, ["ano", "id_municipio", "rede"], "left")
        .select(F.lit("municipio").alias("nivel"), "ano", "id_municipio", nulo.alias("sigla_uf"),
                "serie", "rede", "rede_descricao", "taxa_alfabetizacao", *COLUNAS_META)
    )

    mf = meta_uf.withColumn("rede", codigo_rede_meta()).select("ano", "sigla_uf", "rede", *COLUNAS_META)
    nivel_uf = (
        uf.where(F.col("rede") == "5")
        .join(mf, ["ano", "sigla_uf", "rede"], "left")
        .select(F.lit("uf").alias("nivel"), "ano", nulo.alias("id_municipio"), "sigla_uf",
                "serie", "rede", "rede_descricao", "taxa_alfabetizacao", *COLUNAS_META)
    )

    nivel_brasil = meta_brasil.select(
        F.lit("brasil").alias("nivel"), "ano", nulo.alias("id_municipio"), nulo.alias("sigla_uf"),
        nulo.alias("serie"), codigo_rede_meta().alias("rede"), "rede_descricao",
        "taxa_alfabetizacao", *COLUNAS_META,
    )

    uniao = reduce(lambda a, b: a.unionByName(b),
                   [d.select(*colunas) for d in (nivel_municipio, nivel_uf, nivel_brasil)])
    return aplicar_meta(uniao)


def construir_evolucao_temporal(resultado_vs_meta):
    """Série por UF e Brasil, com variação em p.p. em relação ao ano anterior. Sem projeção (poucos pontos)."""
    from pyspark.sql import Window
    base = resultado_vs_meta.where(F.col("nivel").isin("uf", "brasil"))
    janela = Window.partitionBy("nivel", "sigla_uf").orderBy("ano")
    return (
        base.withColumn("taxa_ano_anterior", F.lag("taxa_alfabetizacao").over(janela))
        .withColumn("variacao_pp_ano_anterior", F.round(F.col("taxa_alfabetizacao") - F.col("taxa_ano_anterior"), 4))
        .withColumn("pontos_na_serie", F.count("ano").over(Window.partitionBy("nivel", "sigla_uf")))
        .select("nivel", "sigla_uf", "ano", "rede", "rede_descricao", "taxa_alfabetizacao",
                "meta_alfabetizacao_ano", "status_meta", "distancia_ate_meta_pp",
                "variacao_pp_ano_anterior", "pontos_na_serie")
    )


def construir_validacao_cruzada_brasil(alunos, meta_brasil):
    """Brasil (rede pública): taxa recalculada dos microdados × taxa publicada em `meta_alfabetizacao_brasil`."""
    calculada = taxa_calculada_microdados(alunos, por=()).where(F.col("rede") == "5")
    oficial = meta_brasil.withColumn("rede", codigo_rede_meta()).select(
        "ano", "rede", F.col("taxa_alfabetizacao").alias("taxa_oficial_publicada"))
    return (
        calculada.join(oficial, ["ano", "rede"], "left")
        .withColumn("diferenca_pp", F.round(F.col("taxa_alfabetizacao_calculada") - F.col("taxa_oficial_publicada"), 4))
        .withColumn("dentro_tolerancia",
                    F.when(F.col("diferenca_pp").isNull(), F.lit(None).cast("boolean"))
                    .otherwise(F.abs(F.col("diferenca_pp")) <= TOLERANCIA_PP))
        .select("ano", "rede", "alunos_avaliados", "taxa_alfabetizacao_calculada",
                "taxa_oficial_publicada", "diferenca_pp", "dentro_tolerancia")
    )


def construir_features(indicador, meta_municipio):
    """Features em t (ano_features) e alvo em t+1 (ano_alvo). Só entram dados conhecidos até t.

    Excluídas de propósito: taxa, média de português e proporções de nível do ano t+1 (são o próprio alvo ou derivam dele),
    `status_meta`, `diferenca_pp` e `dentro_tolerancia` (derivadas de resultado e validação).
    `id_municipio` é identificador, não feature numérica.
    As proporções por nível (`proporcao_aluno_nivel_*`) ficaram de fora: na fonte só existem para 2024, então seriam
    sempre nulas nas features de 2023. Com mais anos, dá para incluí-las (do ano t).
    """
    atual = indicador.select(
        F.col("ano").alias("ano_features"), "id_municipio", "serie", "rede", "rede_descricao",
        F.col("taxa_alfabetizacao_oficial").alias("taxa_t"),
        F.col("taxa_alfabetizacao_calculada").alias("taxa_calculada_t"),
        F.col("media_portugues_oficial").alias("media_portugues_t"),
        F.col("alunos_avaliados").alias("alunos_avaliados_t"),
    )
    alvo = indicador.where(F.col("taxa_alfabetizacao_oficial").isNotNull()).select(
        (F.col("ano") - 1).alias("ano_features"), "id_municipio", "serie", "rede",
        F.col("ano").alias("ano_alvo"),
        F.col("taxa_alfabetizacao_oficial").alias("alvo_taxa_alfabetizacao"),
    )
    meta_alvo = (
        meta_municipio.withColumn("rede", codigo_rede_meta())
        .select(F.col("ano").alias("ano_features"), "id_municipio", "rede",
                meta_do_ano("ano", deslocamento=1).alias("meta_ano_alvo"))
    )
    chaves = ["ano_features", "id_municipio", "serie", "rede"]
    return atual.join(alvo, chaves, "inner").join(meta_alvo, ["ano_features", "id_municipio", "rede"], "left")

## Regras de qualidade

**Bloqueantes** (reprovou, nada é publicado): tabela não vazia, chave sem duplicata, taxa/meta dentro de [0, 100], `status_meta` só com os três valores, rede de meta sem mapeamento = 0 e **taxa de correspondência dos joins ≥ limiar**.

**Informativas** (só registram): percentual dentro da tolerância na validação cruzada, linhas sem recálculo, contagem por `status_meta` (e `sem_meta > 0`, que prova que o terceiro estado funciona) e consistência entre a taxa das tabelas de meta e a das tabelas de resultado.

In [ ]:
SCHEMA_QUALIDADE = StructType([
    StructField("id_execucao", StringType()), StructField("data_execucao", TimestampType()),
    StructField("entidade", StringType()), StructField("regra", StringType()),
    StructField("tipo", StringType()), StructField("valor_medido", DoubleType()),
    StructField("limite", DoubleType()), StructField("passou", BooleanType()),
    StructField("detalhe", StringType()),
])


def regra(nome, tipo, medido, limite, passou, detalhe):
    return dict(regra=nome, tipo=tipo, valor_medido=float(medido),
                limite=None if limite is None else float(limite), passou=bool(passou), detalhe=detalhe)


def registrar_qualidade(entidade, resultados):
    """Grava as regras numa tabela Delta, que sobrevive ao fim da execução."""
    linhas = [(ID_EXECUCAO, DATA_EXECUCAO, entidade, r["regra"], r["tipo"], r["valor_medido"],
               r["limite"], r["passou"], r["detalhe"]) for r in resultados]
    (spark.createDataFrame(linhas, schema=SCHEMA_QUALIDADE).write.format(FORMATO_TABELA).mode("append")
     .saveAsTable(tabela_gold("_quality_results")))


def exigir(entidade, resultados):
    """Levanta erro se alguma regra BLOQUEANTE reprovou. Não há try/except: a falha interrompe a execução."""
    falhas = [r for r in resultados if r["tipo"] == "bloqueante" and not r["passou"]]
    if falhas:
        resumo = "; ".join(f"{r['regra']}={r['valor_medido']:.4g}" for r in falhas)
        raise RuntimeError(f"[{entidade}] regras bloqueantes reprovadas: {resumo}")


def regras_basicas(df, chaves, colunas_taxa=()):
    total = df.count()
    unicas = df.select(*chaves).distinct().count()
    res = [
        regra("linhas_maior_que_zero", "bloqueante", total, 1, total > 0, "linhas na tabela"),
        regra("chave_sem_duplicata", "bloqueante", total - unicas, 0, total == unicas, f"chave {list(chaves)}"),
    ]
    for c in colunas_taxa:
        fora = df.where(F.col(c).isNotNull() & ((F.col(c) < 0) | (F.col(c) > 100))).count()
        res.append(regra(f"valor_entre_0_e_100:{c}", "bloqueante", fora, 0, fora == 0, "valores fora de [0, 100]"))
    return res


def taxa_correspondencia(esquerda, direita, chaves):
    """Parte das linhas da esquerda que encontram par na direita (join que não casa vira nulo, não erro)."""
    pares = direita.select(*chaves).distinct().withColumn("_achou", F.lit(1))
    j = esquerda.select(*chaves).join(pares, list(chaves), "left")
    total = j.count()
    casados = j.where(F.col("_achou").isNotNull()).count()
    return (casados / total if total else 0.0), casados, total


def regras_dos_joins(silver):
    """Rodam ANTES de construir/publicar a Gold. Taxas abaixo do limiar são bloqueantes."""
    res = []
    calculada = taxa_calculada_microdados(silver["alunos"])
    mm = silver["meta_alfabetizacao_municipio"].withColumn("rede", codigo_rede_meta())
    mf = silver["meta_alfabetizacao_uf"].withColumn("rede", codigo_rede_meta())

    sem_mapa = (silver["meta_alfabetizacao_municipio"].where(codigo_rede_meta().isNull()).count()
                + silver["meta_alfabetizacao_uf"].where(codigo_rede_meta().isNull()).count()
                + silver["meta_alfabetizacao_brasil"].where(codigo_rede_meta().isNull()).count())
    res.append(regra("rede_de_meta_sem_mapeamento", "bloqueante", sem_mapa, 0, sem_mapa == 0,
                     "linhas de meta com rede fora do mapa REDE_META_PARA_CODIGO"))

    joins = [
        ("municipio -> microdados recalculados", silver["municipio"], calculada, ["ano", "id_municipio", "serie", "rede"]),
        ("municipio(rede 3) -> meta_alfabetizacao_municipio", silver["municipio"].where(F.col("rede") == "3"), mm,
         ["ano", "id_municipio", "rede"]),
        ("uf(rede 5) -> meta_alfabetizacao_uf", silver["uf"].where(F.col("rede") == "5"), mf, ["ano", "sigla_uf", "rede"]),
    ]
    for nome, esq, dir_, chaves in joins:
        taxa, casados, total = taxa_correspondencia(esq, dir_, chaves)
        res.append(regra(f"taxa_correspondencia:{nome}", "bloqueante", taxa, LIMIAR_JOIN, taxa >= LIMIAR_JOIN,
                         f"{casados} de {total} linhas casam"))

    # Informativa: linhas de meta sem resultado correspondente.
    for nome, meta, resultado, chaves in [
        ("meta_alfabetizacao_municipio sem municipio(rede 3)", mm,
         silver["municipio"].where(F.col("rede") == "3"), ["ano", "id_municipio", "rede"]),
        ("meta_alfabetizacao_uf sem uf(rede 5)", mf, silver["uf"].where(F.col("rede") == "5"), ["ano", "sigla_uf", "rede"]),
    ]:
        taxa, casados, total = taxa_correspondencia(meta, resultado, chaves)
        res.append(regra(f"meta_sem_resultado:{nome}", "informativa", total - casados, None, True,
                         f"{total - casados} de {total} linhas de meta sem resultado (ex.: anos sem microdado)"))

    # Informativa: a taxa das tabelas de meta confere com a das tabelas de resultado? (valida o mapeamento de rede)
    for nome, meta, resultado, chaves in [
        ("meta_municipio x municipio", mm, silver["municipio"], ["ano", "id_municipio", "rede"]),
        ("meta_uf x uf", mf, silver["uf"], ["ano", "sigla_uf", "rede"]),
    ]:
        j = (meta.select(*chaves, F.col("taxa_alfabetizacao").alias("t_meta"))
             .join(resultado.select(*chaves, F.col("taxa_alfabetizacao").alias("t_res")), chaves, "inner")
             .where(F.col("t_meta").isNotNull() & F.col("t_res").isNotNull()))
        total = j.count()
        ok = j.where(F.abs(F.col("t_meta") - F.col("t_res")) <= TOLERANCIA_PP).count()
        res.append(regra(f"consistencia_taxa:{nome}", "informativa", ok / total if total else 0.0, None, True,
                         f"{ok} de {total} pares dentro de {TOLERANCIA_PP} p.p."))
    return res


def regras_extras(nome, df):
    res = []
    if nome == "resultado_vs_meta":
        estados = {r["status_meta"]: r["count"] for r in df.groupBy("status_meta").count().collect()}
        invalidos = sum(v for k, v in estados.items() if k not in ("atingiu", "nao_atingiu", "sem_meta"))
        res.append(regra("status_meta_so_tres_valores", "bloqueante", invalidos, 0, invalidos == 0,
                         "valores fora de atingiu / nao_atingiu / sem_meta"))
        for k in ("atingiu", "nao_atingiu", "sem_meta"):
            res.append(regra(f"linhas_status_meta:{k}", "informativa", estados.get(k, 0), None, True,
                             "sem_meta > 0 prova que o terceiro estado funciona" if k == "sem_meta" else "contagem"))
    if nome in ("indicador_municipio", "validacao_cruzada_brasil"):
        ok = df.where(F.col("dentro_tolerancia")).count()
        com_calc = df.where(F.col("dentro_tolerancia").isNotNull()).count()
        res.append(regra("recalculada_dentro_da_tolerancia", "informativa", ok / com_calc if com_calc else 0.0, None, True,
                         f"{ok} de {com_calc} linhas com |recalculada - oficial| <= {TOLERANCIA_PP} p.p."))
        sem_calc = df.where(F.col("dentro_tolerancia").isNull()).count()
        res.append(regra("linhas_sem_recalculo", "informativa", sem_calc, None, True,
                         "sem microdado correspondente (taxa recalculada nula)"))
    if nome == "features_alfabetizacao":
        alvo_nulo = df.where(F.col("alvo_taxa_alfabetizacao").isNull()).count()
        res.append(regra("alvo_sem_nulo", "bloqueante", alvo_nulo, 0, alvo_nulo == 0, "linhas sem alvo"))
        com_meta = df.where(F.col("meta_ano_alvo").isNotNull()).count()
        res.append(regra("linhas_com_meta_do_ano_alvo", "informativa", com_meta, None, True, "meta nula é preservada, não vira zero"))
    return res


def publicar_gold(nome, df, chaves, colunas_taxa=()):
    """Gold -> staging -> regras -> (reprovou: para) -> promove."""
    staging, destino = tabela_gold(f"_staging_{nome}"), tabela_gold(nome)
    df.write.format(FORMATO_TABELA).mode("overwrite").option("overwriteSchema", "true").saveAsTable(staging)
    stg = spark.table(staging)
    resultados = regras_basicas(stg, chaves, colunas_taxa) + regras_extras(nome, stg)
    linhas = int(resultados[0]["valor_medido"])
    resultados.append(regra("linhas_saida", "informativa", linhas, None, True, "linhas publicadas na Gold"))
    registrar_qualidade(nome, resultados)
    exigir(nome, resultados)
    spark.sql(f"CREATE OR REPLACE TABLE {destino} AS SELECT * FROM {staging}")
    spark.sql(f"DROP TABLE {staging}")
    return linhas

## Execução da rodada

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOGO}`.`{GOLD_SCHEMA}`")

ID_EXECUCAO = str(uuid.uuid4())
DATA_EXECUCAO = datetime.now(timezone.utc)
print(f"Execução {ID_EXECUCAO} iniciada em {DATA_EXECUCAO.isoformat()}")

NOMES_SILVER = ["alunos", "municipio", "uf", "dicionario", "meta_alfabetizacao_brasil",
                "meta_alfabetizacao_uf", "meta_alfabetizacao_municipio"]
silver = {nome: spark.table(tabela_silver(nome)) for nome in NOMES_SILVER}

# 1) Regras dos joins: bloqueantes, ANTES de construir qualquer Gold.
resultados_joins = regras_dos_joins(silver)
registrar_qualidade("joins", resultados_joins)
for r in resultados_joins:
    print(f"{'OK ' if r['passou'] else 'FALHA'} [{r['tipo']}] {r['regra']}: {r['valor_medido']:.4g}  ({r['detalhe']})")
exigir("joins", resultados_joins)

# 2) Constrói e publica cada tabela (a próxima só roda se a anterior passou nas regras bloqueantes).
indicador = construir_indicador_municipio(silver["municipio"], silver["alunos"])
n = publicar_gold("indicador_municipio", indicador, ["ano", "id_municipio", "serie", "rede"],
                  ["taxa_alfabetizacao_oficial", "taxa_alfabetizacao_calculada"])
print(f"indicador_municipio: {n} linhas")

resultado = construir_resultado_vs_meta(silver["municipio"], silver["uf"], silver["meta_alfabetizacao_brasil"],
                                        silver["meta_alfabetizacao_uf"], silver["meta_alfabetizacao_municipio"])
n = publicar_gold("resultado_vs_meta", resultado, ["nivel", "ano", "id_municipio", "sigla_uf", "serie", "rede"],
                  ["taxa_alfabetizacao", "meta_alfabetizacao_ano"])
print(f"resultado_vs_meta: {n} linhas")

evolucao = construir_evolucao_temporal(spark.table(tabela_gold("resultado_vs_meta")))
n = publicar_gold("evolucao_temporal", evolucao, ["nivel", "sigla_uf", "ano"], ["taxa_alfabetizacao"])
print(f"evolucao_temporal: {n} linhas")

validacao = construir_validacao_cruzada_brasil(silver["alunos"], silver["meta_alfabetizacao_brasil"])
n = publicar_gold("validacao_cruzada_brasil", validacao, ["ano", "rede"],
                  ["taxa_alfabetizacao_calculada", "taxa_oficial_publicada"])
print(f"validacao_cruzada_brasil: {n} linhas")

features = construir_features(spark.table(tabela_gold("indicador_municipio")), silver["meta_alfabetizacao_municipio"])
n = publicar_gold("features_alfabetizacao", features, ["ano_features", "id_municipio", "serie", "rede"],
                  ["taxa_t", "alvo_taxa_alfabetizacao"])
print(f"features_alfabetizacao: {n} linhas")

## Verificação

In [ ]:
print("Regras desta execução:")
display(spark.table(tabela_gold("_quality_results")).where(F.col("id_execucao") == ID_EXECUCAO)
        .select("entidade", "regra", "tipo", "valor_medido", "limite", "passou", "detalhe")
        .orderBy("entidade", "tipo", "regra"))

print("resultado_vs_meta: contagem por nível e status (sem_meta deve ser > 0):")
display(spark.table(tabela_gold("resultado_vs_meta")).groupBy("nivel", "ano", "status_meta").count()
        .orderBy("nivel", "ano", "status_meta"))

print("Validação cruzada (Brasil, rede pública): taxa recalculada dos microdados x taxa publicada")
display(spark.table(tabela_gold("validacao_cruzada_brasil")).orderBy("ano"))

print("evolucao_temporal (Brasil):")
display(spark.table(tabela_gold("evolucao_temporal")).where(F.col("nivel") == "brasil").orderBy("ano"))

## Demonstração de falha (o pipeline precisa mostrar que **para** quando a regra reprova)

Esta célula **não toca nas tabelas Gold**. Ela corrompe uma cópia em memória (uma taxa de 150%), roda as mesmas regras bloqueantes e registra a reprovação em `gold._quality_results` (entidade `demo_falha`). O erro é capturado **só aqui**, de propósito, para a demonstração; no fluxo real ele interrompe a execução.

In [ ]:
corrompida = spark.table(tabela_gold("resultado_vs_meta")).limit(100).withColumn(
    "taxa_alfabetizacao", F.lit(150.0))  # dado ruim proposital: taxa acima de 100%

resultados_demo = regras_basicas(
    corrompida, ["nivel", "ano", "id_municipio", "sigla_uf", "serie", "rede"], ["taxa_alfabetizacao"])
registrar_qualidade("demo_falha", resultados_demo)

try:
    exigir("demo_falha", resultados_demo)
    raise AssertionError("A demonstração deveria ter falhado e não falhou.")
except RuntimeError as erro:
    print("FALHA ESPERADA (prova de que regra bloqueante interrompe):")
    print(erro)

## Exportação do dataset de features para a Fase 3

O desafio pede guardar a base analítica para o modelo preditivo da Fase 3. A tabela Delta `gold.features_alfabetizacao` já persiste; esta célula também exporta um **Parquet** para um *volume* do Unity Catalog. Se o seu workspace não permitir criar volumes, o erro aparece aqui (e só aqui) e as tabelas Gold continuam publicadas.

In [ ]:
spark.sql(f"CREATE VOLUME IF NOT EXISTS `{CATALOGO}`.`{GOLD_SCHEMA}`.`exportacoes`")
CAMINHO_EXPORTACAO = f"/Volumes/{CATALOGO}/{GOLD_SCHEMA}/exportacoes/features_alfabetizacao"
(spark.table(tabela_gold("features_alfabetizacao")).write.mode("overwrite").parquet(CAMINHO_EXPORTACAO))
print(f"Parquet exportado em {CAMINHO_EXPORTACAO}")
print(f"Linhas lidas de volta: {spark.read.parquet(CAMINHO_EXPORTACAO).count()}")

## Linhagem (Gold ← Silver)

| Gold | Silver de origem |
|---|---|
| `indicador_municipio` | `municipio` (taxa oficial), `alunos` (taxa recalculada); a decodificação de rede/série veio do `dicionario` na Silver |
| `resultado_vs_meta` | `municipio`, `uf`, `meta_alfabetizacao_municipio`, `meta_alfabetizacao_uf`, `meta_alfabetizacao_brasil` |
| `evolucao_temporal` | `resultado_vs_meta` (e, portanto, `uf`, `meta_alfabetizacao_uf`, `meta_alfabetizacao_brasil`) |
| `validacao_cruzada_brasil` | `alunos`, `meta_alfabetizacao_brasil` |
| `features_alfabetizacao` | `indicador_municipio` (e, portanto, `municipio`, `alunos`), `meta_alfabetizacao_municipio` |

As sete tabelas da fonte aparecem na linhagem: `dicionario` entra pela decodificação feita na Silver.